# Frequent itemsets on all baskets

Moved unchanged from `steam_cleaning.ipynb` (only the imports below are new). The baskets are the played games from `data_preparation.ipynb`; this notebook mines all of them to explore the data and the algorithms' cost, while the recommender in `recommender.ipynb` mines the training baskets only.

In [1]:
import time

import numpy as np
import pandas as pd

## Frequent itemsets (course method)

The baskets come from the shared data definition in `recommender/data.py`, which every method uses: one basket per user holding the games they actually **played** (owned-but-unplayed games are left out: bundles and giveaways make ownership weak evidence of taste). Games need at least 50 players and users at least 5 played games; the two filters repeat until both hold. The table shows the counts after each step.

Here we mine all filtered baskets to explore the data. The recommender in `recommender.ipynb` mines the **training** baskets only, so the games hidden for testing never count towards a rule.

In [2]:
from recommender import data, itemsets, rules

interactions = data.build_interactions(data.load_user_items())
display(interactions.filter_log)
all_baskets = itemsets.baskets(interactions.playtime)
game_name = lambda i: interactions.item_names[interactions.item_ids[i]]

,users,games,played pairs
step,,,
played entries,68403,10050,3246352
filter pass 1,59717,3720,3146315
filter pass 2,59716,3718,3146213
filter pass 3,59716,3718,3146213


### PCY for pairs, Apriori for triples

PCY counts single games and hashes every pair into a bucket in pass 1; in pass 2 it only counts pairs whose two games are frequent and whose bucket is frequent. Triples use the Apriori step: a triple is only counted if all three of its pairs are frequent (Leskovec, Rajaraman & Ullman, *Mining of Massive Datasets*, sections 6.2–6.3). Both are implemented in `recommender/itemsets.py` (compiled with Numba) and checked against brute-force counting in `tests/test_itemsets.py`.

In [3]:
import math

MIN_SUPPORT = 100     # baskets (users) that must contain the itemset
NUM_BUCKETS = 10_000_000

t0 = time.time()
pairs = itemsets.pcy_pairs(all_baskets, MIN_SUPPORT, NUM_BUCKETS)
pair_time = time.time() - t0
t0 = time.time()
triples = itemsets.apriori_triples(all_baskets, pairs, MIN_SUPPORT)
triple_time = time.time() - t0

n_frequent_items = int((pairs.item_support >= MIN_SUPPORT).sum())
print(f"{n_frequent_items:,} frequent games")
print(f"PCY: {pairs.n_candidates:,} candidate pairs, {pairs.frequent.sum() // 2:,} frequent pairs ({pair_time:.1f} s)")
print(f"Apriori: {triples.n_candidates:,} candidate triples, {len(triples.support):,} frequent triples ({triple_time:.1f} s)")
print(f"(all triples of frequent games: {math.comb(n_frequent_items, 3):,})")

2,657 frequent games
PCY: 481,658 candidate pairs, 315,978 frequent pairs (1.7 s)
Apriori: 39,356,028 candidate triples, 9,435,843 frequent triples (180.7 s)
(all triples of frequent games: 3,122,718,960)


In [4]:
upper = np.triu(pairs.pair_support * pairs.frequent, k=1)
top = np.dstack(np.unravel_index(np.argsort(-upper, axis=None)[:15], upper.shape))[0]
display(pd.DataFrame(
    [(game_name(a), game_name(b), int(upper[a, b])) for a, b in top],
    columns=["Game 1", "Game 2", "Played together (users)"],
))

order = np.argsort(-triples.support)[:10]
display(pd.DataFrame(
    [(*map(game_name, triples.items[i]), int(triples.support[i])) for i in order],
    columns=["Game 1", "Game 2", "Game 3", "Played together (users)"],
))

,Game 1,Game 2,Played together (users)
0,Garry's Mod,Counter-Strike: Global Offensive,31207
1,Unturned,Garry's Mod,26382
2,Left 4 Dead 2,Counter-Strike: Global Offensive,26194
3,Garry's Mod,Left 4 Dead 2,25697
4,Unturned,Counter-Strike: Global Offensive,25063
5,Terraria,Garry's Mod,22904
6,Terraria,Counter-Strike: Global Offensive,21392
7,Unturned,Left 4 Dead 2,19776
8,PAYDAY 2,Counter-Strike: Global Offensive,19011
9,PAYDAY 2,Garry's Mod,18235


,Game 1,Game 2,Game 3,Played together (users)
0,Unturned,Garry's Mod,Counter-Strike: Global Offensive,21539
1,Garry's Mod,Left 4 Dead 2,Counter-Strike: Global Offensive,21479
2,Terraria,Garry's Mod,Counter-Strike: Global Offensive,18606
3,Unturned,Garry's Mod,Left 4 Dead 2,17284
4,Unturned,Left 4 Dead 2,Counter-Strike: Global Offensive,16627
5,PAYDAY 2,Garry's Mod,Counter-Strike: Global Offensive,16283
6,Terraria,Unturned,Garry's Mod,16078
7,Terraria,Garry's Mod,Left 4 Dead 2,15759
8,Counter-Strike: Source,Garry's Mod,Counter-Strike: Global Offensive,15133
9,Terraria,Left 4 Dead 2,Counter-Strike: Global Offensive,15106


### Does the PCY bitmap help at this scale?

PCY only saves work if most hash buckets stay below the support threshold, so that the bitmap rules out pairs. With few buckets every bucket collects enough pairs to be frequent and PCY counts exactly the same candidates as Apriori. The table shows the share of frequent buckets and the candidate pairs PCY keeps for growing hash tables.

In [5]:
saturation = itemsets.bucket_saturation(all_baskets, MIN_SUPPORT, [10_000, 100_000, 1_000_000, 10_000_000, 100_000_000])
# Textbook memory model (MMDS 6.2-6.3) with 4-byte counts; single-game counts are left out (tiny).
# A-Priori, pass 2: every pair of frequent games in a triangular array, 4 bytes per pair.
# PCY, pass 1: a 4-byte counter per bucket. Pass 2: the counters are replaced by a bitmap (1 bit per
# bucket), and the scattered candidates are stored as (i, j, count) triples, 12 bytes each.
saturation["A-Priori pass 2 (MB)"] = saturation["pairs of frequent items"] * 4 / 1e6
saturation["PCY pass 1 (MB)"] = saturation["buckets"] * 4 / 1e6
saturation["PCY pass 2 (MB)"] = (saturation["buckets"] / 8 + saturation["candidate pairs"] * 12) / 1e6
display(saturation.round(3))

,buckets,frequent bucket share,candidate pairs,pairs of frequent items,A-Priori pass 2 (MB),PCY pass 1 (MB),PCY pass 2 (MB)
0,10000,1.000,3528496,3528496,14.114,0.04,42.343
1,100000,1.000,3528496,3528496,14.114,0.40,42.354
2,1000000,0.508,2248777,3528496,14.114,4.00,27.110
3,10000000,0.034,481658,3528496,14.114,40.00,7.030
4,100000000,0.003,331830,3528496,14.114,400.00,16.482


With few buckets every bucket collects enough pairs to be frequent, and PCY keeps every pair of frequent games as a candidate, exactly like A-Priori. Because PCY's candidates are scattered, it cannot use A-Priori's compact triangular array and stores each candidate with its two game ids (12 instead of 4 bytes), so its pass 2 only needs less memory than A-Priori's if the bitmap removes at least two thirds of the candidates. Of the sizes tried, the smallest that does this is ten million buckets, and its counters then need 40 MB in pass 1, more than A-Priori needs in pass 2 (14 MB).

In the textbook setting this is still worthwhile: pass 1 has memory to spare, and PCY helps when A-Priori's pass 2 would not fit in memory. Here, with 2,657 frequent games, A-Priori's pass 2 needs only about 14 MB, so the bitmap brings no practical saving; it would matter for much larger catalogues. (The table is the textbook model, not our code's memory use: for simplicity `itemsets.pcy_pairs` keeps a plain n × n count matrix, which fits easily at this size.) The pruning that does matter is A-Priori's for triples: of the C(n, 3) possible triples, only those whose three pairs are frequent are counted (see the candidate count above).